# AI‑Augmented Control Testing: Jira Change Management

**Purpose:** Build an auditable workflow that tests a change‑management control from Jira tickets by combining deterministic checks with a probabilistic comment classifier.


In [ ]:

# --- Setup & Imports ---
import os, json, re, math, datetime, itertools, uuid
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Tuple
from collections import defaultdict, Counter

import pandas as pd

# Ensure folder structure
os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("Environment ready.", {"cwd": os.getcwd(), "outputs_dir": os.path.abspath("outputs")})


## Learning objectives

By the end of this lab you will be able to:

- Express a SOX control as deterministic test logic with clear pass and fail criteria.  
- Retrieve and structure Jira ticket data for control testing.  
- Classify natural‑language approval comments with an LLM using a fail‑closed prompt pattern.  
- Validate that approvals come from required approvers and decide the control outcome.  
- Produce an audit‑ready evidence record and capture traces for review (LangSmith or Opik).

In [ ]:

# --- Define control & evaluation types ---
@dataclass
class ControlRequirement:
    code: str
    description: str
    required_roles: List[str]  # e.g., ["Dev Manager", "QA Lead"]
    min_approvals: int = 2
    keywords_approve: Tuple[str,...] = ("approved", "approve", "lgtm", "looks good", "sign off", "ship it")
    disqualify_keywords: Tuple[str,...] = ("pending", "block", "reject", "do not", "needs changes")

@dataclass
class EvidenceRecord:
    ticket_id: str
    control_code: str
    inputs: Dict[str, Any]
    deterministic_checks: Dict[str, Any]
    probabilistic_scores: Dict[str, Any]
    decision: str  # "PASS" | "FAIL"
    reason: str
    trace_path: Optional[str] = None

CONTROL = ControlRequirement(
    code="PAY-002",
    description="Changes require at least two approvals including the Dev Manager before merge.",
    required_roles=["Dev Manager", "QA Lead"],
    min_approvals=2
)

print("Control loaded:", CONTROL)


## Prerequisites and inputs

- Access to sample Jira tickets (JSON stubs or API).  
- A small list of required approvers and their identifiers (email, username, or group).  
- Optional: OpenAI or compatible LLM API key for comment classification.  
- Optional: LangSmith or Opik credentials for tracing and evaluation.

You can complete this lab with provided JSON stubs if live API access is not available in the classroom environment.


In [ ]:

# --- Data access layer ---
# In this lab we simulate Jira via local files; if you have a JIRA_* env configured, you can extend the client.

SAMPLE_TICKETS = [
    {
        "id": "PROJ-101",
        "summary": "Enable feature flag for payments",
        "assignee": "alice",
        "status": "Done",
        "fields": {"components": ["payments"], "change_category": "Config"},
        "comments": [
            {"author": "mike", "role": "Dev Manager", "text": "LGTM, approved for release."},
            {"author": "sara", "role": "QA Lead", "text": "Tested on staging, looks good to me."},
        ]
    },
    {
        "id": "PROJ-102",
        "summary": "Bump python to 3.12",
        "assignee": "bob",
        "status": "Done",
        "fields": {"components": ["platform"], "change_category": "Dependency"},
        "comments": [
            {"author": "tim", "role": "Engineer", "text": "ship it"},
            {"author": "jane", "role": "Product Manager", "text": "pending rollout next week"},
        ]
    },
    {
        "id": "PROJ-103",
        "summary": "Refactor payment retry logic",
        "assignee": "carol",
        "status": "In Review",
        "fields": {"components": ["payments"], "change_category": "Code"},
        "comments": [
            {"author": "mike", "role": "Dev Manager", "text": "needs changes before I can approve."},
            {"author": "sara", "role": "QA Lead", "text": "tests failing on edge case"},
        ]
    }
]

def load_tickets_from_csv(path="data/jira_tickets.csv") -> List[Dict[str,Any]]:
    if not os.path.exists(path):
        return []
    df = pd.read_csv(path)
    # Expect columns: id, summary, assignee, status, components, change_category, comments_json
    tickets = []
    for _, r in df.iterrows():
        comments = json.loads(r.get("comments_json","[]"))
        components = json.loads(r.get("components","[]")) if isinstance(r.get("components"), str) and r.get("components","").startswith("[") else [r.get("components")]
        tickets.append({
            "id": r["id"],
            "summary": r.get("summary",""),
            "assignee": r.get("assignee",""),
            "status": r.get("status",""),
            "fields": {"components": components, "change_category": r.get("change_category","Unknown")},
            "comments": comments
        })
    return tickets

def get_jira_tickets(ticket_ids: Optional[List[str]] = None) -> List[Dict[str,Any]]:
    tickets = load_tickets_from_csv() or SAMPLE_TICKETS
    if ticket_ids:
        idset = set(ticket_ids)
        tickets = [t for t in tickets if t["id"] in idset]
    return tickets

print(f"Loaded {len(get_jira_tickets())} tickets (sample or CSV).")


## Architecture overview

```
Ticket IDs  ──>  Jira fetch (deterministic)  ──>  Ticket bundle
                                   │
                                   ▼
                        Completeness validation (deterministic)
                                   │
                                   ▼
                 Comment classification (probabilistic, LLM)
                                   │
                                   ▼
                Approver validation (deterministic, policy)
                                   │
                                   ▼
                     PASS/FAIL decision + rationale
                                   │
                                   ▼
         Evidence record (JSONL) + tracing (LangSmith/Opik)
```

Key idea: keep facts and policy checks deterministic, reserve the LLM for classifying intent in free‑text comments.


In [ ]:

# --- Normalization helpers ---
APPROVER_ROLE_CANON = {
    "dev manager": "Dev Manager",
    "engineering manager": "Dev Manager",
    "qa lead": "QA Lead",
    "quality lead": "QA Lead",
    "product manager": "Product Manager",
    "engineer": "Engineer",
}

def canon_role(role: str) -> str:
    if not role:
        return "Unknown"
    r = role.strip().lower()
    return APPROVER_ROLE_CANON.get(r, role.strip())

def normalize_ticket(t: Dict[str,Any]) -> Dict[str,Any]:
    comments = []
    for c in t.get("comments", []):
        comments.append({
            "author": c.get("author","").strip(),
            "role": canon_role(c.get("role","")),
            "text": c.get("text","").strip()
        })
    return {
        "id": t["id"],
        "summary": t.get("summary",""),
        "assignee": t.get("assignee",""),
        "status": t.get("status",""),
        "components": list(t.get("fields",{}).get("components",[]) or []),
        "change_category": t.get("fields",{}).get("change_category","Unknown"),
        "comments": comments
    }

tickets_norm = [normalize_ticket(t) for t in get_jira_tickets()]
pd.DataFrame([{
    "id": t["id"], "status": t["status"], "category": t["change_category"], 
    "n_comments": len(t["comments"])
} for t in tickets_norm])


## Step 0. Define the control and acceptance criteria

Write the control being tested in plain language, then translate it to exact, testable rules.

**Example control:**  
“All production application changes above threshold T require dual approval by the required approver set before merge or release.”

**Acceptance criteria:**

- Amount or risk threshold is computed or read from the ticket.  
- Required approver set is defined (usernames, groups, or roles).  
- Approvals must be dated before deployment.  
- Approved comments are explicit and attributable to a valid approver.  
- If any element is missing or ambiguous, outcome is **FAIL** with rationale “Insufficient evidence.”


In [ ]:

# --- Deterministic checks ---



## Step 1. Retrieve tickets (deterministic)

Tasks:

- Given a list of Jira ticket IDs, fetch: summary, description, status, labels, created/updated timestamps, assignees, commenters, and all comments.  
- Normalize into a consistent schema for the rest of the pipeline.  
- Redact any PII not required for testing.

Notes:

- If API is unavailable, load provided JSON fixtures from `./data/jira_samples/`.  
- Record `source_system`, `ticket_id`, and `ingested_at` for traceability.


In [ ]:

# --- Probabilistic scoring (rule-based NLP) ---
POSITIVE_PATTERNS = [r"\blgtm\b", r"\bapprove(d)?\b", r"\blooks? good\b", r"\bship it\b", r"\bsign[- ]?off\b"]
NEGATIVE_PATTERNS = [r"\bpending\b", r"\bblock(ed|ing)?\b", r"\breject(ed|ion)?\b", r"\bneeds? changes?\b", r"\bdo not\b"]




## Step 2. Validate ticket completeness (deterministic)

Run schema and field checks before using the LLM:

- Required fields present: `ticket_id`, `created`, `updated`, `comments[]`, `author`, `body`, `timestamp`.  
- Optional policy fields present if used: `risk_score`, `amount`, `component`, `deployment_status`.  
- Drop or flag tickets that fail validation with reason codes such as `missing_comments` or `no_author_ids`.

Output: a clean list of tickets ready for classification.


## Step 3. Identify approval comments (probabilistic)

Design a fail‑closed prompt that classifies each comment with a small, explicit label set.

**Labels:** `approved`, `rejected`, `fyi`, `unclear`

**Output schema per comment:**

```json
{
  "ticket_id": "APP-4321",
  "comment_id": "cmt-0098",
  "label": "approved|rejected|fyi|unclear",
  "rationale": "short explanation using quoted evidence",
  "actor": {"id": "user@company.com", "display": "First Last"},
  "timestamp": "2025-09-12T15:44:00Z"
}
```

Fail‑closed rule: if the comment does not explicitly convey approval, label `unclear` and require deterministic evidence elsewhere.


## Step 4. Validate approvers (deterministic)

- Map commenters to identities using email, username, or SSO mapping tables.  
- Define the required approver set for the control (users, groups, or roles).  
- Check that at least the required set appears among comments labeled `approved`.  
- Validate timestamps relative to deployment or merge time.

Output: a boolean `approver_set_satisfied` with a list of matched approvers.


## Step 5. Decide PASS or FAIL

Combine results into a single decision record per ticket:

- `required_approvers` satisfied.  
- At least one `approved` comment per required approver.  
- Approvals are prior to deployment and within policy window.  
- No contradictory `rejected` comments by required approvers.

Decision policy:

- If any prerequisite check fails or approvals are ambiguous, return **FAIL** with rationale.  
- Otherwise, return **PASS** with citations to specific comment IDs.


## Step 6. Produce an evidence record

Emit one JSON object per ticket with everything needed for audit review.

**Evidence schema (example):**

```json
{
  "ticket_id": "APP-4321",
  "control_id": "AP-127",
  "threshold": 5000,
  "required_approvers": ["vp_finance", "it_manager"],
  "approved_comments": [
    {"comment_id": "c-12", "actor":"vp_finance", "timestamp":"2025-09-12T10:42:00Z"}
  ],
  "approver_set_satisfied": false,
  "decision": "FAIL",
  "rationale": "Missing approval from it_manager",
  "ingested_at": "2025-10-07T16:00:00Z",
  "trace": {"platform":"LangSmith", "run_id":"<id>", "project":"SOX-Audit-Copilot"}
}
```

Store as `./outputs/evidence.jsonl` for batch review.
